# 01 — Auditoria da base de conhecimento

Este notebook inspeciona o corpus utilizado pelo RAG e pelo agente. Ele verifica estrutura, identificadores, origem, licenças e referências. Os registros de ocorrências e as políticas internas são simulados; as demais entradas são paráfrases de fontes públicas.

In [6]:
from pathlib import Path
import json
import sys

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

In [7]:
from collections import Counter

base = json.loads((ROOT / "base/trechos.json").read_text(encoding="utf-8"))
obrigatorios = {"id", "titulo", "texto", "tipo", "referencia", "url", "licenca"}
ids = [item["id"] for item in base]
assert len(ids) == len(set(ids)), "IDs duplicados na base."
assert all(obrigatorios.issubset(item) for item in base), "Campos obrigatórios ausentes."
assert all(item["texto"].strip() for item in base), "Trecho sem conteúdo."
assert all(item["url"].strip() for item in base if item["tipo"] == "publico"), "Fonte pública sem URL."

contagem = Counter(item["tipo"] for item in base)
print(f"Total de trechos: {len(base)} | IDs únicos: {len(set(ids))}")
print("Documentos por tipo:", dict(sorted(contagem.items())))
print("Fontes públicas com URL:", sum(item["tipo"] == "publico" and bool(item["url"]) for item in base))
print("Verificação estrutural: aprovada")

Total de trechos: 17 | IDs únicos: 17
Documentos por tipo: {'politica_simulada': 5, 'publico': 7, 'simulado': 5}
Fontes públicas com URL: 7
Verificação estrutural: aprovada


## Composição, formato e licença

A unidade de recuperação é um trecho registrado em `base/trechos.json`, não um PDF integral. As licenças são os avisos declarados nos metadados do corpus; isso **não** constitui uma auditoria jurídica.

In [8]:
rotulos = {"simulado": "Registros de demonstração", "politica_simulada": "Políticas acadêmicas simuladas", "publico": "Paráfrases de fontes públicas"}
print(f"{'Origem dos trechos':34} {'Formato':8} {'Quantidade':>10}  {'Licença / condição declarada'}")
print("-" * 125)
for tipo in ("simulado", "politica_simulada", "publico"):
    grupo = [item for item in base if item["tipo"] == tipo]
    licencas = sorted({item["licenca"] for item in grupo})
    aviso = "; ".join(licencas)
    print(f"{rotulos[tipo]:34} {'JSON':8} {len(grupo):>10}  {aviso}")

Origem dos trechos                 Formato  Quantidade  Licença / condição declarada
-----------------------------------------------------------------------------------------------------------------------------
Registros de demonstração          JSON              5  Dado fictício produzido para o projeto
Políticas acadêmicas simuladas     JSON              5  Texto autoral, política acadêmica simulada
Paráfrases de fontes públicas      JSON              7  Paráfrase autoral de texto legal público; verificar versão vigente; Paráfrase autoral; conferir atualização municipal; Paráfrase autoral; verificar direitos e atualização da página municipal; Resumo autoral de norma municipal; conferir redação consolidada vigente


## Rastreabilidade

Cada trecho contém ID, título, referência, tipo, URL quando aplicável e aviso de licença. Os identificadores permitem que o assistente aponte a origem utilizada, mas uma citação válida **não garante** que a resposta esteja correta.

In [9]:
print(f"{'ID':6} {'Tipo':19} {'Título':59} Referência")
print("-" * 130)
for item in base:
    titulo = item["titulo"][:57]
    print(f"{item['id']:6} {item['tipo']:19} {titulo:59} {item['referencia']}")

ID     Tipo                Título                                                      Referência
----------------------------------------------------------------------------------------------------------------------------------
S001   simulado            Registro simulado OC001 — buraco na calçada                 PassoLivre, cadastro de demonstração OC001
S002   simulado            Registro simulado OC002 — poste obstruindo passagem         PassoLivre, cadastro de demonstração OC002
S003   simulado            Registro simulado OC003 — desnível rejeitado                PassoLivre, cadastro de demonstração OC003
S004   politica_simulada   Política do PassoLivre — validação dos relatos              PassoLivre, política acadêmica interna (simulada)
S005   politica_simulada   Política do PassoLivre — limites de segurança               PassoLivre, política acadêmica interna (simulada)
S006   politica_simulada   Política do PassoLivre — encerramento de ocorrência         PassoLivre, política

In [10]:
print("Fontes públicas catalogadas:")
for item in base:
    if item["tipo"] == "publico":
        print(f"{item['id']}: {item['url']}")

Fontes públicas catalogadas:
G001: https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2015/lei/l13146.htm
G002: https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2015/lei/l13146.htm
G003: https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2015/lei/l13146.htm
G004: https://prefeitura.sp.gov.br/subprefeituras/w/calcadas/37136
G005: https://prefeitura.sp.gov.br/subprefeituras/w/calcadas/37136
G006: https://legislacao.prefeitura.sp.gov.br/leis/decreto-59671-de-7-de-agosto-de-2020
G007: https://prefeitura.sp.gov.br/web/subprefeituras/w/calcadas/37447


## Conclusão

O corpus contém 17 trechos identificados, com proveniência explícita. Os dados fictícios permitem avaliar status e recusas sem expor endereços ou pessoas reais. A base é pequena e não comprova acessibilidade de locais específicos. A aplicação das normas depende de conferência da versão oficial vigente.